# 02 — Sentiment Analysis

Three sentiment models were applied to the scraped reviews in the full project:

1. **VADER** (NLTK) — rule-based, bag-of-words classifier. Run on both IMDb user reviews and RogerEbert professional reviews. See cell: `VADER classification`.
2. **TextBlob** polarity — used as a simple baseline for exploratory histograms in the EDA notebook.
3. **RoBERTa** (Hugging Face cardiffnlp/twitter-roberta-base-sentiment) — transformer-based, applied with a `split_text()` helper to respect the 512-token limit, then aggregated by majority rule across chunks. The RoBERTa code itself lives in a separate notebook maintained by my teammate and is not reproduced here; this notebook focuses on the VADER run and how labels were constructed.

## Key results from the report

| Model     | True Positive Rate | False Positive Rate |
|-----------|--------------------|---------------------|
| VADER     | 0.784              | 0.434               |
| RoBERTa   | 0.849              | 0.324 (FNR)         |

RoBERTa outperformed VADER, especially on negative-review detection.

## Ground-truth label construction
For classification, we binarized user/critic star ratings:
- **Positive (1):** rating ≥ 6 on IMDb's 0–10 scale
- **Negative (0):** rating ≤ 5

The VADER cell in Evan's original notebook used a 6.5 threshold on one genre while running classification. The report describes using a ≥ 6 threshold across the full dataset — we use the report's convention in this cleaned version.

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import nltk

from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

nltk.download('vader_lexicon', quiet=True)

USER_DIR = "../data/user_reviews"
PRO_DIR  = "../data/professional_reviews"
GENRES = ["adventure", "comedy", "crime", "horror", "superhero"]


## Load the data

In [ ]:
def load_all_user_reviews():
    frames = []
    for g in GENRES:
        df = pd.read_csv(f"{USER_DIR}/{g}.csv")
        df["Genre"] = g
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def load_all_pro_reviews():
    frames = []
    for g in GENRES:
        df = pd.read_csv(f"{PRO_DIR}/{g}.csv")
        df["Genre"] = g
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


user_df = load_all_user_reviews()
pro_df  = load_all_pro_reviews()

print(f"User reviews: {len(user_df):,}")
print(f"Professional reviews: {len(pro_df):,}")
user_df.head()


## VADER classification

Build binary ground-truth labels from the star rating, then compare against VADER's polarity scores.

In [ ]:
# Ground-truth labels from star ratings (>= 6 is positive)
user_df["Positive"] = (user_df["Review Score"] >= 6).astype(int)

analyzer = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    """Return 1 if VADER flags the review as predominantly positive, else 0."""
    if not isinstance(text, str) or not text.strip():
        return 0
    scores = analyzer.polarity_scores(text)
    return int(scores["pos"] > 0.22)

user_df["VADER"] = user_df["Review Text"].apply(vader_sentiment)
user_df[["Title", "Review Score", "Positive", "VADER"]].head()


### Confusion matrix & metrics

In [ ]:
matrix = confusion_matrix(user_df["Positive"], user_df["VADER"])
cm = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=[0, 1])
cm.plot()
plt.title("VADER vs. Ground Truth (IMDb user reviews)")
plt.show()

print(classification_report(user_df["Positive"], user_df["VADER"],
                            target_names=["Negative", "Positive"]))


### Per-genre breakdown

Useful to see which genres VADER handles well and where it struggles.

In [ ]:
rows = []
for g in GENRES:
    sub = user_df[user_df["Genre"] == g]
    if sub.empty:
        continue
    m = confusion_matrix(sub["Positive"], sub["VADER"], labels=[0, 1])
    tn, fp, fn, tp = m.ravel() if m.shape == (2, 2) else (0, 0, 0, 0)
    tpr = tp / (tp + fn) if (tp + fn) else float("nan")
    fpr = fp / (fp + tn) if (fp + tn) else float("nan")
    rows.append({
        "Genre": g.title(),
        "n": len(sub),
        "Positives (truth)": int(sub["Positive"].sum()),
        "TPR": round(tpr, 3),
        "FPR": round(fpr, 3),
    })

pd.DataFrame(rows)


## Notes on RoBERTa

The RoBERTa half of the analysis (contributed by Roy) used `cardiffnlp/twitter-roberta-base-sentiment` from Hugging Face. Because reviews regularly exceed RoBERTa's 512-token input limit, the pipeline was:

1. `split_text(review)` → list of token chunks within the limit
2. Classify each chunk (positive / neutral / negative)
3. Aggregate by majority rule across chunks
4. Collapse to binary: {positive, neutral} → 1, {negative} → 0

Reported results from the final paper (see `../report/final_report.pdf`):

- **True Positive Rate:** 0.849
- **False Negative Rate:** 0.324

RoBERTa caught substantially more true negatives than VADER, which was expected: VADER's bag-of-words approach misses the longer contextual cues that matter in movie reviews (sarcasm, contrastive phrases, etc.), while RoBERTa's transformer architecture handles them natively.

The RoBERTa notebook itself lives in a separate file outside this repo; if you want it here, contact me.

## Summary

- **VADER:** serviceable for catching clearly positive reviews but weak on negatives (high FPR)
- **RoBERTa:** stronger across the board, especially on negatives
- **Takeaway:** for review classification where negative sentiment matters, transformer models are worth the extra compute